# Exercise 3 — Amazon Bedrock Guardrail: PII Blocking

This notebook demonstrates how Amazon Bedrock Guardrails detect and block (or anonymize) **Personally Identifiable Information (PII)** before the model responds.

| PII Type | Action |
|---|---|
| EMAIL | BLOCK |
| PHONE | BLOCK |
| US_SOCIAL_SECURITY_NUMBER | BLOCK |
| CREDIT_DEBIT_CARD_NUMBER | BLOCK |
| NAME | ANONYMIZE |
| ADDRESS | ANONYMIZE |

**Guardrail ID**: `4yuhcf0hhpfd`  **Version**: `1`  **Region**: `us-east-1`

## 1. Setup

In [ ]:
import boto3
import json

GUARDRAIL_ID      = "4yuhcf0hhpfd"
GUARDRAIL_VERSION = "1"
MODEL_ID          = "anthropic.claude-3-haiku-20240307-v1:0"
REGION            = "us-east-1"

client = boto3.client("bedrock-runtime", region_name=REGION)
print("Client ready.", "Region:", REGION)

## 2. Helper — Invoke Model with Guardrail

In [ ]:
def invoke(prompt: str) -> dict:
    """Send a prompt through the guardrail and return action + output."""
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 256,
        "messages": [{"role": "user", "content": prompt}],
    }
    response = client.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        trace="ENABLED",
    )
    result = json.loads(response["body"].read())
    output = (
        result["content"][0]["text"]
        if result.get("content")
        else "[No content returned]"
    )
    guardrail_action = response["ResponseMetadata"]["HTTPHeaders"].get(
        "x-amzn-bedrock-guardrail-action", "NONE"
    )
    return {
        "stop_reason": result.get("stop_reason"),
        "guardrail_action": guardrail_action,
        "output": output,
    }

print("Helper defined.")

## 3. Test: EMAIL — should be BLOCKED

In [ ]:
result = invoke("My email address is john.doe@example.com. Can you confirm it?")
print("Guardrail Action :", result["guardrail_action"])
print("Stop Reason      :", result["stop_reason"])
print("Output           :", result["output"])

## 4. Test: PHONE — should be BLOCKED

In [ ]:
result = invoke("Please call me at 555-867-5309 to discuss my account.")
print("Guardrail Action :", result["guardrail_action"])
print("Stop Reason      :", result["stop_reason"])
print("Output           :", result["output"])

## 5. Test: SSN — should be BLOCKED

In [ ]:
result = invoke("My social security number is 123-45-6789.")
print("Guardrail Action :", result["guardrail_action"])
print("Stop Reason      :", result["stop_reason"])
print("Output           :", result["output"])

## 6. Test: Credit Card — should be BLOCKED

In [ ]:
result = invoke("Charge my card number 4111 1111 1111 1111 for the order.")
print("Guardrail Action :", result["guardrail_action"])
print("Stop Reason      :", result["stop_reason"])
print("Output           :", result["output"])

## 7. Test: Name + Address — should be ANONYMIZED

In [ ]:
result = invoke("Hi, I am Alice Johnson and I live at 742 Evergreen Terrace, Springfield.")
print("Guardrail Action :", result["guardrail_action"])
print("Stop Reason      :", result["stop_reason"])
print("Output           :", result["output"])

## 8. Test: No PII — should PASS through

In [ ]:
result = invoke("What is the capital of France?")
print("Guardrail Action :", result["guardrail_action"])
print("Stop Reason      :", result["stop_reason"])
print("Output           :", result["output"])

## 9. Run All Tests in Batch

In [ ]:
test_cases = [
    ("Email — BLOCKED",       "My email address is john.doe@example.com. Can you confirm it?"),
    ("Phone — BLOCKED",       "Please call me at 555-867-5309 to discuss my account."),
    ("SSN — BLOCKED",         "My social security number is 123-45-6789."),
    ("Credit Card — BLOCKED", "Charge my card number 4111 1111 1111 1111 for the order."),
    ("Name+Address — ANON",   "Hi, I am Alice Johnson and I live at 742 Evergreen Terrace, Springfield."),
    ("No PII — PASS",         "What is the capital of France?"),
]

print(f"{'Test':<30} {'Guardrail Action':<20} {'Stop Reason'}")
print("-" * 70)
for label, prompt in test_cases:
    r = invoke(prompt)
    print(f"{label:<30} {r['guardrail_action']:<20} {r['stop_reason']}")

## Summary

| Scenario | Expected Guardrail Action | Expected Stop Reason |
|---|---|---|
| EMAIL in prompt | INTERVENED | guardrail_intervened |
| PHONE in prompt | INTERVENED | guardrail_intervened |
| SSN in prompt | INTERVENED | guardrail_intervened |
| Credit card in prompt | INTERVENED | guardrail_intervened |
| NAME + ADDRESS | INTERVENED (anonymized) | guardrail_intervened |
| No PII | NONE | end_turn |

When the guardrail **intervenes**, the model never sees the PII — Bedrock short-circuits the call and returns the configured `blockedInputMessaging` string instead of a model response.